In [2]:
import pandas as pd
from pybaseball import pitching_stats, chadwick_register
from pybaseball import cache

# add cache layer to skip re-download in second run
cache.enable()

In [8]:
from playwright.async_api import async_playwright

async def fetch_fangraphs():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()

        # land on the real page first so Cloudflare clears
        await page.goto(
            "https://www.fangraphs.com/leaders/major-league",
            wait_until="domcontentloaded",
            timeout=60000,
        )
        await page.wait_for_timeout(10000)

        # now hit the API with the cleared session
        resp = await page.goto(
            "https://www.fangraphs.com/api/leaders/major-league/data"
            "?pos=all&stats=pit&lg=all&qual=0&type=c"
            "&season=2024&month=0&season1=2024&ind=1"
            "&team=0&rost=0&age=0&page=1_50",
            wait_until="domcontentloaded",
            timeout=60000,
        )
        text = await page.inner_text("body")
        await browser.close()

    return text

text = await fetch_fangraphs()
print(text[:500])

In [11]:
from playwright.async_api import async_playwright

async def fetch_fangraphs():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()

        await page.goto(
            "https://www.fangraphs.com/leaders/major-league",
            wait_until="domcontentloaded",
            timeout=60000,
        )
        await page.wait_for_selector("table", timeout=60000)

        resp = await page.goto(
            "https://www.fangraphs.com/api/leaders/major-league/data"
            "?pos=all&stats=pit&lg=all&qual=0&type=c"
            "&season=2024&month=0&season1=2024&ind=1"
            "&team=0&rost=0&age=0&page=1_50",
            wait_until="domcontentloaded",
            timeout=60000,
        )
        text = await page.inner_text("body")
        await browser.close()

    return text

text = await fetch_fangraphs()
print(text[:500])

TargetClosedError: Page.wait_for_selector: Target page, context or browser has been closed
Call log:
  - waiting for locator("table") to be visible


In [17]:
from playwright.async_api import async_playwright
from playwright_stealth import Stealth
import json
import asyncio
import pandas as pd

YEAR_RANGE = (2015, 2025)

async def fetch_fangraphs_all():
    stealth = Stealth()
    start, end = YEAR_RANGE
    years = list(range(start, end + 1))

    print(f"Years to pull: {years}")

    async with stealth.use_async(async_playwright()) as p:
        browser = await p.chromium.launch(
            headless=False,
            args=["--disable-blink-features=AutomationControlled"],
        )
        page = await browser.new_page()

        all_dfs = []
        for year in years:
            print(f"  Pulling {year}...")

            api_future = asyncio.get_event_loop().create_future()

            async def capture_api(response, fut=api_future):
                if "/api/leaders/major-league/data" in respons
                    try:
                        body = await response.json()
                        fut.set_result(body)
                    except:
                        pass

            page.on("response", capture_api)

            await page.goto(
                "https://www.fangraphs.com/leaders/major-leagu
                f"?month=0&pos=all&stats=pit&type=8"
                f"&season1={year}&season={year}"
                "&pageitems=2000000000",
                wait_until="domcontentloaded",
                timeout=120000,
            )

            if year == years[0]:
                print(">>> Click the checkbox / open new tab if needed <<<")

            try:
                chunk = await asyncio.wait_for(api_future, tim
            except asyncio.TimeoutError:
                print(f"  {year}: timed out waiting for API re
                page.remove_listener("response", capture_api)
                continue

            page.remove_listener("response", capture_api)

            if isinstance(chunk, list):
                all_dfs.append(pd.DataFrame(chunk))
            elif isinstance(chunk, dict) and "data" in chunk:
                all_dfs.append(pd.DataFrame(chunk["data"]))
            else:
                print(f"  {year}: unexpected format")
                continue

            print(f"  Got {len(all_dfs[-1])} rows")

        await browser.close()

    return pd.concat(all_dfs, ignore_index=True)

df_fangraphs = await fetch_fangraphs_all()
print(df_fangraphs.shape)
print([c for c in df_fangraphs.columns if "WAR" in c.upper()])
print(df_fangraphs[["PlayerName", "Season", "WAR"]].head(10))    

SyntaxError: unterminated string literal (detected at line 40) (3986467964.py, line 40)

In [3]:
from playwright.async_api import async_playwright
from playwright_stealth import Stealth
import json
import asyncio
import pandas as pd

YEAR_RANGE = (2015, 2025)

async def fetch_fangraphs_year(year):
    stealth = Stealth()

    async with stealth.use_async(async_playwright()) as p:
        browser = await p.chromium.launch(
            headless=False,
            args=["--disable-blink-features=AutomationControlled"],
        )
        page = await browser.new_page()

        await page.goto(
            "https://www.fangraphs.com/leaders/major-league"
            f"?month=0&pos=all&stats=pit&type=8"
            f"&season1={year}&season={year}"
            "&pageitems=2000000000",
            wait_until="domcontentloaded",
            timeout=120000,
        )

        input(f"  >>> {year}: table loaded? Press ENTER to scr

        api_future = asyncio.get_event_loop().create_future()

        async def capture_api(response, fut=api_future):
            if "/api/leaders/major-league/data" in response.url and not fut.done():
                try:
                    body = await response.json()
                    if isinstance(body, list) and len(body) >
                        fut.set_result(body)
                    elif isinstance(body, dict) and "data" in 0:
                        fut.set_result(body)
                except:
                    pass

        page.on("response", capture_api)

        # reload to trigger the API call now that we're listening
        await page.reload(wait_until="domcontentloaded", timeo

        chunk = await asyncio.wait_for(api_future, timeout=120

        await browser.close()

    return chunk

async def fetch_fangraphs_all():
    start, end = YEAR_RANGE
    years = list(range(start, end + 1))

    print(f"Years to pull: {years}")

    all_dfs = []
    for year in years:
        print(f"\n  === {year} ===")

        chunk = await fetch_fangraphs_year(year)

        if isinstance(chunk, list):
            all_dfs.append(pd.DataFrame(chunk))
        elif isinstance(chunk, dict) and "data" in chunk:
            all_dfs.append(pd.DataFrame(chunk["data"]))
        else:
            print(f"  {year}: unexpected format, skipping")
            continue

        print(f"  {year}: got {len(all_dfs[-1])} rows")

    return pd.concat(all_dfs, ignore_index=True)

df_fangraphs = await fetch_fangraphs_all()
print(df_fangraphs.shape)
print([c for c in df_fangraphs.columns if "WAR" in c.upper()])
print(df_fangraphs[["PlayerName", "Season", "WAR"]].head(10)) 

SyntaxError: unterminated f-string literal (detected at line 28) (2470209850.py, line 28)

In [1]:
from playwright.async_api import async_playwright
from playwright_stealth import Stealth
import json
import asyncio
import pandas as pd

YEAR_RANGE = (2015, 2025)

async def fetch_fangraphs_year(year):
    stealth = Stealth()

    async with stealth.use_async(async_playwright()) as p:
        browser = await p.chromium.launch(
            headless=False,
            args=["--disable-blink-features=AutomationControlled"],
        )
        page = await browser.new_page()

        await page.goto(
            "https://www.fangraphs.com/leaders/major-league"
            f"?month=0&pos=all&stats=pit&type=8"
            f"&pageitems=2000000000&startdate=&enddate="
            f"&season1={year}&season={year}",
            wait_until="domcontentloaded",
            timeout=120000,
        )

        input(f"  >>> {year}: table loaded? Press ENTER to scrape <<<")

        api_future = asyncio.get_event_loop().create_future()

        async def capture_api(response, fut=api_future):
            if "/api/leaders/major-league/data" in response.url and not fut.done():
                try:
                    body = await response.json()
                    if isinstance(body, list) and len(body) > 0:
                        fut.set_result(body)
                    elif isinstance(body, dict) and "data" in body and len(body["data"]) > 0:
                        fut.set_result(body)
                except:
                    pass

        page.on("response", capture_api)

        # reload to trigger the API call now that we're listening
        await page.reload(wait_until="domcontentloaded", timeout=120000)

        chunk = await asyncio.wait_for(api_future, timeout=120)

        await browser.close()

    return chunk

async def fetch_fangraphs_all():
    start, end = YEAR_RANGE
    years = list(range(start, end + 1))

    print(f"Years to pull: {years}")

    all_dfs = []
    for year in years:
        print(f"\n  === {year} ===")

        chunk = await fetch_fangraphs_year(year)

        if isinstance(chunk, list):
            all_dfs.append(pd.DataFrame(chunk))
        elif isinstance(chunk, dict) and "data" in chunk:
            all_dfs.append(pd.DataFrame(chunk["data"]))
        else:
            print(f"  {year}: unexpected format, skipping")
            continue

        print(f"  {year}: got {len(all_dfs[-1])} rows")

    return pd.concat(all_dfs, ignore_index=True)

df_fangraphs = await fetch_fangraphs_all()
print(df_fangraphs.shape)
print([c for c in df_fangraphs.columns if "WAR" in c.upper()])
print(df_fangraphs[["PlayerName", "Season", "WAR"]].head(10))

Years to pull: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

  === 2015 ===


TargetClosedError: Page.reload: Target page, context or browser has been closed